# Phase II_01: Spectral Relabeling with USGS MTBS Standards

**Objective**: Create multi-class burn labels using official USGS burn severity classifications

**Approach**: Classify each image independently using single-image NBR (not change-based dNBR)

**Input**: CaBuAr dataset with pre-fire and post-fire Sentinel-2 images

**Output**: Multi-class labels (7 classes) for both pre and post images; combined for training

**QC**: Verify post-fire shows MORE severe burns than pre-fire

**Reference**: USGS MTBS standards for single-timepoint NBR classification

## Setup: Google Drive and Dependencies

In [ ]:
import sys
import torch
import numpy as np
from pathlib import Path
from datetime import datetime
import json

sys.path.insert(0, '/content/RETINNA/src')

# Mount Google Drive
print("📁 Initializing Google Drive...")
from drive_utils import setup_drive_for_colab

drive_manager = setup_drive_for_colab(verbose=True)

if drive_manager is None:
    raise RuntimeError("Failed to connect to Google Drive - aborting")

print("\n" + "="*70)
print("PHASE II_01: SPECTRAL RELABELING WITH USGS MTBS STANDARDS")
print("="*70)
print(f"Session start: {datetime.now().isoformat()}")
print(f"Drive cache: {drive_manager.root}")

## Load Dataset

In [ ]:
%pip install -q torchgeo torch torchvision matplotlib numpy
from dataset import get_dataloaders
from colab_utils import setup_cabuaur_cached

# Setup Google Drive caching if on Colab
root = None
try:
    root = setup_cabuaur_cached()
    print("✓ Google Drive caching enabled")
except (ImportError, RuntimeError) as e:
    print(f"⚠ Drive caching failed: {e}")
    print("Using default TorchGeo cache")
    root = '/tmp/cabuaur'

# Load dataloaders (all splits)
print("\n📊 Loading CaBuAr dataset...")
dataloaders = get_dataloaders(batch_size=1, num_workers=0, normalize=True, root=root)

train_dataset = dataloaders['datasets']['train']
val_dataset = dataloaders['datasets']['val']
test_dataset = dataloaders['datasets']['test']

print(f"✓ Train samples: {len(train_dataset)}")
print(f"✓ Val samples: {len(val_dataset)}")
print(f"✓ Test samples: {len(test_dataset)}")
print(f"✓ Total: {len(train_dataset) + len(val_dataset) + len(test_dataset)} samples")

## Spectral Analysis: Compute USGS NBR and Classification

In [ ]:
def compute_nbr(image):
    """
    Compute NBR for a single Sentinel-2 image.

    Args:
        image: [12, H, W] Sentinel-2 bands in CaBuAr order
        Bands: B01(0), B02(1), B03(2), B04(3), B05(4), B06(5), B07(6), B08(7), B8A(8), B09(9), B11(10), B12(11)

    Returns:
        dict with NBR, MNDWI, Blue, NIR
    """
    # Correct band indices for CaBuAr
    nir_idx, swir1_idx, green_idx, blue_idx = 7, 10, 2, 1

    nir = image[nir_idx]      # B08 (NIR)
    swir = image[swir1_idx]   # B11 (SWIR-1)
    green = image[green_idx]  # B03 (Green)
    blue = image[blue_idx]    # B02 (Blue)

    eps = 1e-8

    # NBR = (NIR - SWIR) / (NIR + SWIR)
    nbr = (nir - swir) / (nir + swir + eps)

    # MNDWI = (Green - SWIR) / (Green + SWIR)
    mndwi = (green - swir) / (green + swir + eps)

    return {
        'nbr': nbr,
        'mndwi': mndwi,
        'blue': blue,
        'nir': nir
    }


def classify_nbr(nbr, mndwi, blue, nir):
    """
    Classify pixels based on single-image NBR values.

    Thresholds are for single-timepoint NBR (not dNBR change).
    Classes: 0=Unburned, 1=Low, 2=Moderate, 3=High, 4=Extreme, 5=Water, 6=Cloud

    Args:
        nbr: [H, W] NBR tensor
        mndwi: [H, W] MNDWI tensor
        blue: [H, W] B02 (blue) reflectance
        nir: [H, W] B08 (NIR) reflectance

    Returns:
        [H, W] uint8 tensor with class labels (0-6)
    """
    labels = torch.zeros_like(nbr, dtype=torch.uint8)

    # Step 1: Cloud detection (highest priority)
    cloud_mask = (blue > 0.25) & ((blue / (nir + 1e-8)) > 0.8)
    labels[cloud_mask] = 6

    # Step 2: Water detection
    water_mask = (mndwi > 0.3) & ~cloud_mask
    labels[water_mask] = 5

    # Step 3: Burn severity (USGS MTBS thresholds for single NBR)
    non_special = ~cloud_mask & ~water_mask

    # Extreme Severity: NBR < -0.27
    labels[(nbr < -0.27) & non_special] = 4

    # High Severity: -0.27 ≤ NBR ≤ -0.1
    labels[(nbr >= -0.27) & (nbr <= -0.1) & non_special] = 3

    # Moderate Severity: -0.1 < NBR ≤ 0.05
    labels[(nbr > -0.1) & (nbr <= 0.05) & non_special] = 2

    # Low Severity: 0.05 < NBR ≤ 0.27
    labels[(nbr > 0.05) & (nbr <= 0.27) & non_special] = 1

    # Unburned: NBR ≥ 0.27 (already initialized to 0)

    return labels


print("✓ Spectral analysis functions defined (correct band indices: B08/B11)")

## Process All Samples: Train + Val + Test

In [ ]:
def process_dataset(dataset, dataset_name):
    """
    Process dataset and generate labels for both pre and post images.
    
    Each image classified independently using single-image NBR thresholds.
    QC check: post-fire should show MORE severe burns than pre-fire.
    
    Returns:
        labels_pre: [N, 512, 512] pre-fire labels
        labels_post: [N, 512, 512] post-fire labels
        metrics: dict with distributions and QC results
    """
    print(f"\n📊 Processing {dataset_name} set ({len(dataset)} samples)...")
    
    all_labels_pre = []
    all_labels_post = []
    
    class_counts_pre = {i: 0 for i in range(7)}
    class_counts_post = {i: 0 for i in range(7)}
    
    qc_checks = {
        'total_pixels': 0,
        'post_more_severe': 0,
        'post_equal_severity': 0,
        'post_less_severe': 0
    }
    
    for sample_idx in range(len(dataset)):
        if (sample_idx + 1) % max(1, len(dataset) // 10) == 0:
            print(f"  {sample_idx + 1}/{len(dataset)} samples processed...")
        
        sample = dataset[sample_idx]
        image = sample['image'].numpy()  # [2, 12, 512, 512]
        
        pre_fire = image[0]
        post_fire = image[1]
        
        # Compute NBR for pre-fire
        indices_pre = compute_nbr(pre_fire)
        nbr_pre = torch.from_numpy(indices_pre['nbr']).float()
        mndwi_pre = torch.from_numpy(indices_pre['mndwi']).float()
        blue_pre = torch.from_numpy(indices_pre['blue']).float()
        nir_pre = torch.from_numpy(indices_pre['nir']).float()
        
        labels_pre = classify_nbr(nbr_pre, mndwi_pre, blue_pre, nir_pre)
        labels_pre = labels_pre.numpy()
        
        # Compute NBR for post-fire
        indices_post = compute_nbr(post_fire)
        nbr_post = torch.from_numpy(indices_post['nbr']).float()
        mndwi_post = torch.from_numpy(indices_post['mndwi']).float()
        blue_post = torch.from_numpy(indices_post['blue']).float()
        nir_post = torch.from_numpy(indices_post['nir']).float()
        
        labels_post = classify_nbr(nbr_post, mndwi_post, blue_post, nir_post)
        labels_post = labels_post.numpy()
        
        # QC check: post should be more burned than pre
        more_severe = (labels_post > labels_pre).sum()
        equal_severity = (labels_post == labels_pre).sum()
        less_severe = (labels_post < labels_pre).sum()
        
        qc_checks['total_pixels'] += labels_pre.size
        qc_checks['post_more_severe'] += more_severe
        qc_checks['post_equal_severity'] += equal_severity
        qc_checks['post_less_severe'] += less_severe
        
        # Store labels
        all_labels_pre.append(labels_pre)
        all_labels_post.append(labels_post)
        
        # Track class distributions
        for class_id in range(7):
            class_counts_pre[class_id] += (labels_pre == class_id).sum()
            class_counts_post[class_id] += (labels_post == class_id).sum()
    
    # Stack labels
    labels_pre_array = np.stack(all_labels_pre, axis=0)  # [N, 512, 512]
    labels_post_array = np.stack(all_labels_post, axis=0)  # [N, 512, 512]
    
    # Compute metrics
    total_pixels = labels_pre_array.size
    metrics = {
        'dataset': dataset_name,
        'n_samples': len(dataset),
        'total_pixels': int(total_pixels),
        'pre_fire': {'class_distribution': {}},
        'post_fire': {'class_distribution': {}},
        'qc_check': {
            'post_more_severe_pct': 100.0 * qc_checks['post_more_severe'] / qc_checks['total_pixels'],
            'post_equal_pct': 100.0 * qc_checks['post_equal_severity'] / qc_checks['total_pixels'],
            'post_less_severe_pct': 100.0 * qc_checks['post_less_severe'] / qc_checks['total_pixels']
        }
    }
    
    class_names = [
        'Unburned',
        'Low Severity',
        'Moderate Severity',
        'High Severity',
        'Extreme Severity',
        'Water',
        'Cloud/Shadow'
    ]
    
    print(f"\n  Pre-fire distribution:")
    for class_id in range(7):
        count = int(class_counts_pre[class_id])
        pct = 100.0 * count / total_pixels
        metrics['pre_fire']['class_distribution'][class_names[class_id]] = {
            'pixels': count,
            'percentage': float(f"{pct:.2f}")
        }
        print(f"    Class {class_id} ({class_names[class_id]:20}): {count:12,} px ({pct:5.2f}%)")
    
    print(f"\n  Post-fire distribution:")
    for class_id in range(7):
        count = int(class_counts_post[class_id])
        pct = 100.0 * count / total_pixels
        metrics['post_fire']['class_distribution'][class_names[class_id]] = {
            'pixels': count,
            'percentage': float(f"{pct:.2f}")
        }
        print(f"    Class {class_id} ({class_names[class_id]:20}): {count:12,} px ({pct:5.2f}%)")
    
    print(f"\n  QC Check (post-fire vs pre-fire severity):")
    print(f"    Post more severe: {metrics['qc_check']['post_more_severe_pct']:.2f}%")
    print(f"    Post equal severity: {metrics['qc_check']['post_equal_pct']:.2f}%")
    print(f"    Post less severe: {metrics['qc_check']['post_less_severe_pct']:.2f}% ⚠️ (should be minimal)")
    
    return labels_pre_array, labels_post_array, metrics


# Process all splits
train_pre, train_post, train_metrics = process_dataset(train_dataset, "Train")
val_pre, val_post, val_metrics = process_dataset(val_dataset, "Val")
test_pre, test_post, test_metrics = process_dataset(test_dataset, "Test")

print("\n✓ All datasets processed")

## Combine and Save Results

In [ ]:
# Combine both pre and post labels for training (doubles training data)
all_labels = np.concatenate([
    train_pre, train_post,
    val_pre, val_post,
    test_pre, test_post
], axis=0)

all_metrics = {
    'timestamp': datetime.now().isoformat(),
    'note': 'Labels computed per image using single-image NBR (not change-based dNBR). Pre and post combined for training.',
    'total_samples': len(train_dataset) + len(val_dataset) + len(test_dataset),
    'total_images': 2 * (len(train_dataset) + len(val_dataset) + len(test_dataset)),
    'splits': {
        'train': train_metrics,
        'val': val_metrics,
        'test': test_metrics
    }
}

print(f"\n💾 Saving results...")
print(f"  Labels shape: {all_labels.shape} (pre + post combined)")
print(f"  Total images for training: {all_metrics['total_images']}")

# Save multi-class labels to Drive
labels_path = drive_manager.save_with_timestamp(
    torch.from_numpy(all_labels),
    rel_path="phase2/II_01_relabeling",
    filename_base="multi_class_labels",
    file_format=".pt",
    verbose=True
)

# Save metrics to Drive
metrics_path = drive_manager.save_with_timestamp(
    all_metrics,
    rel_path="phase2/II_01_relabeling",
    filename_base="metrics",
    file_format=".json",
    verbose=True
)

if labels_path and metrics_path:
    print(f"\n✓ Results saved to Drive")
    print(f"  Labels: {labels_path.name}")
    print(f"  Metrics: {metrics_path.name}")
else:
    print(f"\n✗ Failed to save to Drive")

## Quality Control: Verify Results

In [ ]:
# Reload from Drive to verify
print("\n🔍 Quality Control: Verifying saved files...")

latest_labels = drive_manager.get_latest_file(
    rel_path="phase2/II_01_relabeling",
    filename_pattern="multi_class_labels"
)

if latest_labels:
    loaded_labels = torch.load(latest_labels)
    print(f"✓ Labels verified: {loaded_labels.shape}")
    
    # Check class distribution
    print(f"\nClass distribution in loaded labels:")
    class_names = [
        'Unburned',
        'Low Severity',
        'Moderate Severity',
        'High Severity',
        'Extreme Severity',
        'Water',
        'Cloud/Shadow'
    ]
    
    for class_id in range(7):
        count = (loaded_labels == class_id).sum().item()
        total = loaded_labels.numel()
        pct = 100.0 * count / total
        print(f"  {class_id}: {class_names[class_id]:20} {count:12,} px ({pct:5.2f}%)")
else:
    print(f"✗ Could not verify saved labels")

print(f"\n✓ Phase II_01 complete at {datetime.now().isoformat()}")
print(f"\nNext: II_02_rgb_ir_training.ipynb")